In [1]:
%pip install -U transformers safetensors

   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.8 MB 10.3 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.8 MB 13.8 MB/s eta 0:00:01
   ----------------------- ---------------- 7.6/12.8 MB 13.2 MB/s eta 0:00:01
   -------------------------------- ------- 10.5/12.8 MB 13.3 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.8 MB 13.6 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 12.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------------------------- -------------- 1.8/2.9 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 9.8 MB/s eta 0:00:00

   ----- ---------------------------------- 1/7 [safetensors]
   ----------- ---------------------------- 2/7 [regex]
   ----------- ---------------------------- 2/7 [regex]
   ---------------------- ----------------- 4/7 [typer]
   ----


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
# Chuẩn bị dữ liệu VinaFood21
MODEL_NAME = "microsoft/resnet-50"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

print(processor)

In [ ]:
mean = processor.image_mean
std = processor.image_std

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
# Thay bằng đường dẫn dữ liệu thực tế trên máy
DATA_DIR = r"C:\duong_dan_thuc_te\VinaFood21"

train_folder = os.path.join(DATA_DIR, "train")
val_folder = os.path.join(DATA_DIR, "val")

full_train_aug = datasets.ImageFolder(
    train_folder,
    transform=train_transform
)

full_train_eval = datasets.ImageFolder(
    train_folder,
    transform=eval_transform
)

test_ds = datasets.ImageFolder(
    val_folder,
    transform=eval_transform
)

assert len(full_train_aug.classes) == 21
assert full_train_aug.class_to_idx == full_train_eval.class_to_idx
assert full_train_aug.class_to_idx == test_ds.class_to_idx

classes = full_train_aug.classes

n_train = int(0.9 * len(full_train_aug))

generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(full_train_aug),
    generator=generator
).tolist()

train_indices = indices[:n_train]
val_indices = indices[n_train:]

train_ds = Subset(full_train_aug, train_indices)
val_ds = Subset(full_train_eval, val_indices)

train_dl = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True
)

val_dl = DataLoader(
    val_ds,
    batch_size=16,
    shuffle=False
)

test_dl = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Classes:", classes)

In [ ]:
# Load Pretrained ResNet50 và thay classifier
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ResNetForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=21,
    ignore_mismatched_sizes=True,
    id2label={i: name for i, name in enumerate(classes)},
    label2id={name: i for i, name in enumerate(classes)}
)

model = model.to(device)

print(model)

In [ ]:
model.eval()

with torch.no_grad():
    x = torch.randn(2, 3, 224, 224).to(device)

    outputs = model(pixel_values=x)

    print("Output shape:", outputs.logits.shape)

In [ ]:
# Khởi tạo Adam Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

EPOCHS = 10

In [ ]:
# Hàm training
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)
        logits = outputs.logits

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
# Hàm Validation
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=images)
        logits = outputs.logits

        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
# Finetuning Loop
train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model, train_dl, criterion, optimizer, device
    )

    val_loss, val_acc = validate(
        model, val_dl, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_pretrained_resnet50.pth"
        )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

In [ ]:
# Đánh giá Accuracy, Precision, Recall và F1-macro
model.load_state_dict(
    torch.load(
        "best_pretrained_resnet50.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_dl:
        images = images.to(device)

        logits = model(pixel_values=images).logits
        preds = logits.argmax(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-macro:  {f1:.4f}")

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(21)),
    target_names=classes,
    zero_division=0
))

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pretrained ResNet50 - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Pretrained ResNet50 - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=list(range(21)),
    display_labels=classes,
    cmap="Blues",
    xticks_rotation=90,
    ax=ax
)

plt.title("Pretrained ResNet50 - VinaFood21 Confusion Matrix")
plt.tight_layout()
plt.show()